In [1]:
import json
import shutil
import warnings
from pathlib import Path

from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter('ignore', InterpolationWarning)

import config
from input.input import load_raw_data
from model import generics, hybrid_system_exp, grid_search_exp
from model.feature_selection import TimeSeriesFeatureSelector
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from utils.compare_fs_vs_baseline import build_comparison
from utils.export_metrics_to_csv import save_csv

%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Projetos\mestrado_codigos\experiments\src\model\hybrid_system_exp.py' for module 'model.hybrid_system_exp': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Projetos\mestrado_codigos\experiments\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\joaol\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File 

In [2]:
# === Notebook de FS na janela de 10% (pct10) -- ARIMA-SVR hibrido (Additive) / rfecv ===
# Mesma estrutura dos chamados_v4_fs_* ('auto'), mas: (a) lag_size_override =
# resolve_lag_size_pct(N - test_size, 0.10) por serie; (b) comparacao par-a-par
# SEMPRE contra o baseline pct10 DA PROPRIA FAMILIA (celula final), NUNCA
# contra o baseline 'auto'. experiment_id/model_name distintos, sem underscore.
# SVR deterministico -- sem seed. force=False. Unica acao: Restart Kernel -> Run All.
model = Pipeline([
    ('selector', TimeSeriesFeatureSelector(strategy='rfecv')),
    ('estimator', SVR(max_iter=100000)),
])

series_list = ['airlines.txt', 'austres.txt', 'coloradoRiver.txt', 'sunspot.txt', 'windspeedfortaleza.txt', 'samurec.txt']

experiment_id = 'chamados_pct10_fs_arimasvr_rfecv'
model_name = 'aspct10rfecv'   # -> 1aspct10rfecv.pkl (sem underscore, RUNBOOK.md 7)
normalize = True
force = False
model_exec = 1

experiment_params = {
    'linear_model_name': '1arima',
    'diff_kpss': False,
    'horizon': 1,
}

model_parameters = {
    'estimator__C': [10, 100, 1000],
    'estimator__gamma': ['auto'],
    'estimator__kernel': ['rbf'],
    'estimator__epsilon': [0.1, 0.01, 0.001],
    'estimator__tol': [0.001],
}

# Comparacao par-a-par: baseline pct10 da PROPRIA familia (Parte 1). Nunca 'auto'.
baseline_experiment_id = 'chamados_pct10'
baseline_model_name = '1aspct10'
linear_model_name_to_exclude = '1arima'

experiment_dir = Path(config.MODEL_DATA_PATH) / experiment_id
experiment_dir_results = Path(config.ROOT_PATH) / 'results' / experiment_id

In [3]:
# Sanity-check (mesmo padrao dos notebooks 'auto'): Pipeline.get_params(deep=True)
# expoe as chaves que GridSearch vai usar; e strategy <-> experiment_id/model_name
# consistentes entre si.
params = model.get_params(deep=True)
required_keys = {'selector__strategy', 'estimator__C', 'estimator__kernel'}
missing = required_keys - params.keys()
assert not missing, f'get_params(deep=True) nao expos: {missing}'

strategy_slug = model.named_steps['selector'].strategy.replace('_', '')
assert strategy_slug in experiment_id, f'{strategy_slug!r} nao em experiment_id={experiment_id!r}'
assert strategy_slug in model_name, f'{strategy_slug!r} nao em model_name={model_name!r}'
print(f'OK -- strategy={model.named_steps["selector"].strategy!r} consistente; keys expostas.')

OK -- strategy='rfecv' consistente; keys expostas.


In [4]:
# Additive precisa do ARIMA pre-treinado sob o MESMO experiment_id. O ARIMA
# NAO depende de lag_size (auto_arima ajusta sobre a serie original, nao a
# janela) -- copia byte-a-byte de chamados/, o mesmo .pkl usado pela matriz
# 'auto'. Idempotente.
experiment_dir.mkdir(parents=True, exist_ok=True)
for base_name in series_list:
    serie = base_name.split('.')[0]
    src = Path(config.MODEL_DATA_PATH) / 'chamados' / f'{serie}_1arima.pkl'
    dst = experiment_dir / f'{serie}_1arima.pkl'
    assert src.exists(), f'ARIMA pre-treinado ausente: {src} -- rode arima_exec.ipynb antes.'
    shutil.copy(src, dst)
    print(f'{serie}: {src.name} -> {dst}')

airlines: airlines_1arima.pkl -> C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10_fs_arimasvr_rfecv\airlines_1arima.pkl
austres: austres_1arima.pkl -> C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10_fs_arimasvr_rfecv\austres_1arima.pkl
coloradoRiver: coloradoRiver_1arima.pkl -> C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10_fs_arimasvr_rfecv\coloradoRiver_1arima.pkl
sunspot: sunspot_1arima.pkl -> C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10_fs_arimasvr_rfecv\sunspot_1arima.pkl
windspeedfortaleza: windspeedfortaleza_1arima.pkl -> C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10_fs_arimasvr_rfecv\windspeedfortaleza_1arima.pkl
samurec: samurec_1arima.pkl -> C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10_fs_arimasvr_rfecv\samurec_1arima.pkl


In [5]:
# Janela de 10%: lag_size_override = resolve_lag_size_pct(N - test_size, 0.10)
# por serie -- MESMA base que get_max_lag_to_consider (PACF sobre
# ts_univariate[0:-test_size]). Computado aqui, nada a editar.
# force=False: execution() (correcao de 2026-09-02) pula .pkl ja existente
# e nao-vazio -- re-Run All e idempotente.
lag_pct_por_serie = {}
for base_name in series_list:
    n_raw = len(load_raw_data(base_name))
    n_train = n_raw - int(config.TEST_SIZE * n_raw)
    lag_pct = grid_search_exp.resolve_lag_size_pct(n_train, pct=0.10)
    lag_pct_por_serie[base_name] = lag_pct
    print(f'{base_name}  N={n_raw}  N-test={n_train}  lag_pct={lag_pct}')
    exec_gs = grid_search_exp.GridSearch(
        hybrid_system_exp.Additive,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
        lag_size_override=lag_pct,
    )
    exec_gs.execution()

airlines.txt  N=144  N-test=130  lag_pct=13
[skip] C:\Projetos\mestrado_codigos\experiments/data/result/chamados_pct10_fs_arimasvr_rfecv/airlines_1aspct10rfecv.pkl ja existe e force=False -- pulando
austres.txt  N=89  N-test=81  lag_pct=8
[skip] C:\Projetos\mestrado_codigos\experiments/data/result/chamados_pct10_fs_arimasvr_rfecv/austres_1aspct10rfecv.pkl ja existe e force=False -- pulando
coloradoRiver.txt  N=744  N-test=670  lag_pct=67
[skip] C:\Projetos\mestrado_codigos\experiments/data/result/chamados_pct10_fs_arimasvr_rfecv/coloradoRiver_1aspct10rfecv.pkl ja existe e force=False -- pulando
sunspot.txt  N=288  N-test=260  lag_pct=26
[skip] C:\Projetos\mestrado_codigos\experiments/data/result/chamados_pct10_fs_arimasvr_rfecv/sunspot_1aspct10rfecv.pkl ja existe e force=False -- pulando
windspeedfortaleza.txt  N=144  N-test=130  lag_pct=13
[skip] C:\Projetos\mestrado_codigos\experiments/data/result/chamados_pct10_fs_arimasvr_rfecv/windspeedfortaleza_1aspct10rfecv.pkl ja existe e force

In [6]:
from utils.export_metrics_to_csv import run_export_metrics_to_csv

df_metrics = run_export_metrics_to_csv(
    experiment_dir, experiment_dir_results / 'metrics.csv', detail=True,
)
df_metrics

[INFO] 12 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10_fs_arimasvr_rfecv'.

  OK  airlines_1arima.pkl  ->  1 linha(s)
  OK  airlines_1aspct10rfecv.pkl  ->  1 linha(s)
  OK  austres_1arima.pkl  ->  1 linha(s)
  OK  austres_1aspct10rfecv.pkl  ->  1 linha(s)
  OK  coloradoRiver_1arima.pkl  ->  1 linha(s)
  OK  coloradoRiver_1aspct10rfecv.pkl  ->  1 linha(s)
  OK  samurec_1arima.pkl  ->  1 linha(s)
  OK  samurec_1aspct10rfecv.pkl  ->  1 linha(s)
  OK  sunspot_1arima.pkl  ->  1 linha(s)
  OK  sunspot_1aspct10rfecv.pkl  ->  1 linha(s)
  OK  windspeedfortaleza_1arima.pkl  ->  1 linha(s)
  OK  windspeedfortaleza_1aspct10rfecv.pkl  ->  1 linha(s)

[OK] CSV agregado (média das repetições) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_pct10_fs_arimasvr_rfecv\metrics.csv
     12 linha(s) × 20 coluna(s)

[OK] CSV detalhado (por repetição) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_pct10_fs_arim

,ExperimentID,Serie,Modelo,N_Repeticoes,MSE_mean,MSE_std,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,theil_mean,theil_std,ARV_mean,ARV_std,IA_mean,IA_std,POCID_mean,POCID_std
0,chamados_pct10_fs_arimasvr_rfecv,airlines,1arima,1,389.194049,NaN,19.728002,NaN,15.102607,NaN,3.315326,NaN,0.134713,NaN,0.066301,NaN,0.983139,NaN,78.571429,NaN
1,chamados_pct10_fs_arimasvr_rfecv,airlines,1aspct10rfecv,1,444.152942,NaN,21.074936,NaN,18.257246,NaN,3.902291,NaN,0.157941,NaN,0.077026,NaN,0.980536,NaN,78.571429,NaN
2,chamados_pct10_fs_arimasvr_rfecv,austres,1arima,1,358.130880,NaN,18.924346,NaN,13.710292,NaN,0.078380,NaN,0.129713,NaN,0.034904,NaN,0.990999,NaN,75.000000,NaN
3,chamados_pct10_fs_arimasvr_rfecv,austres,1aspct10rfecv,1,567.984695,NaN,23.832429,NaN,16.996673,NaN,0.097270,NaN,0.183812,NaN,0.056740,NaN,0.985495,NaN,75.000000,NaN
4,chamados_pct10_fs_arimasvr_rfecv,coloradoRiver,1arima,1,0.104961,NaN,0.323976,NaN,0.262984,NaN,28.817193,NaN,0.755655,NaN,0.698832,NaN,0.667249,NaN,52.702703,NaN
5,chamados_pct10_fs_arimasvr_rfecv,coloradoRiver,1aspct10rfecv,1,0.151145,NaN,0.388774,NaN,0.307193,NaN,33.144447,NaN,0.786363,NaN,0.807519,NaN,0.586569,NaN,50.000000,NaN
6,chamados_pct10_fs_arimasvr_rfecv,samurec,1arima,1,48.481589,NaN,6.962872,NaN,5.858474,NaN,20.580083,NaN,28.356862,NaN,34.562540,NaN,0.193628,NaN,64.406780,NaN
7,chamados_pct10_fs_arimasvr_rfecv,samurec,1aspct10rfecv,1,52.858049,NaN,7.270354,NaN,6.048188,NaN,20.498476,NaN,3.932152,NaN,5.776597,NaN,0.377423,NaN,59.322034,NaN
8,chamados_pct10_fs_arimasvr_rfecv,sunspot,1arima,1,365.478709,NaN,19.117497,NaN,15.719171,NaN,36.503112,NaN,0.346894,NaN,0.193887,NaN,0.951367,NaN,67.857143,NaN
9,chamados_pct10_fs_arimasvr_rfecv,sunspot,1aspct10rfecv,1,572.121813,NaN,23.919068,NaN,19.050548,NaN,41.973149,NaN,0.344543,NaN,0.202075,NaN,0.937137,NaN,82.142857,NaN


In [7]:
from utils.export_selected_features import run_export_selected_features

df_features = run_export_selected_features(
    experiment_dir, experiment_dir_results / 'selected_features.csv', detail=True,
)
df_features

[INFO] 12 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10_fs_arimasvr_rfecv'.

  OK  airlines_1arima.pkl  ->  sem seletor -- ignorado
  OK  airlines_1aspct10rfecv.pkl  ->  1 linha(s)
  OK  austres_1arima.pkl  ->  sem seletor -- ignorado
  OK  austres_1aspct10rfecv.pkl  ->  1 linha(s)
  OK  coloradoRiver_1arima.pkl  ->  sem seletor -- ignorado
  OK  coloradoRiver_1aspct10rfecv.pkl  ->  1 linha(s)
  OK  samurec_1arima.pkl  ->  sem seletor -- ignorado
  OK  samurec_1aspct10rfecv.pkl  ->  1 linha(s)
  OK  sunspot_1arima.pkl  ->  sem seletor -- ignorado
  OK  sunspot_1aspct10rfecv.pkl  ->  1 linha(s)
  OK  windspeedfortaleza_1arima.pkl  ->  sem seletor -- ignorado
  OK  windspeedfortaleza_1aspct10rfecv.pkl  ->  1 linha(s)

[OK] CSV agregado (média/desvio por série × modelo) gerado em: C:\Projetos\mestrado_codigos\experiments\results\chamados_pct10_fs_arimasvr_rfecv\selected_features.csv
     6 linha(s) × 8 coluna(s)

[OK] CSV detalhado (

,ExperimentID,Serie,Modelo,Strategy,N_Features_Selected_mean,N_Features_Selected_std,N_Repeticoes,N_Features_Total
0,chamados_pct10_fs_arimasvr_rfecv,airlines,1aspct10rfecv,rfecv,12.0,NaN,1,13
1,chamados_pct10_fs_arimasvr_rfecv,austres,1aspct10rfecv,rfecv,7.0,NaN,1,8
2,chamados_pct10_fs_arimasvr_rfecv,coloradoRiver,1aspct10rfecv,rfecv,45.0,NaN,1,67
3,chamados_pct10_fs_arimasvr_rfecv,samurec,1aspct10rfecv,rfecv,88.0,NaN,1,107
4,chamados_pct10_fs_arimasvr_rfecv,sunspot,1aspct10rfecv,rfecv,18.0,NaN,1,26
5,chamados_pct10_fs_arimasvr_rfecv,windspeedfortaleza,1aspct10rfecv,rfecv,8.0,NaN,1,13


In [8]:
# Comparacao PAR-A-PAR: FS pct10 x baseline pct10 da MESMA familia.
# NUNCA contra o baseline 'auto' -- isola o efeito da selecao de features
# do efeito da definicao de janela. A chave do dict e so o rotulo da coluna
# de saida (o slug da estrategia).
fs_dirs = {experiment_id.rsplit('_', 1)[-1]: experiment_dir}
df_cmp = build_comparison(
    Path(config.MODEL_DATA_PATH) / baseline_experiment_id,
    fs_dirs,
    baseline_model_name=baseline_model_name,
    linear_model_name_to_exclude=linear_model_name_to_exclude,
)
save_csv(df_cmp, experiment_dir_results / 'comparison.csv',
         label='comparacao FS pct10 x baseline pct10 (par-a-par)')
df_cmp

[INFO] 30 arquivo(s) .pkl encontrado(s) em 'C:\Projetos\mestrado_codigos\experiments\data\result\chamados_pct10'.

  OK  airlines_1amv1pct10.pkl  ->  10 linha(s)
  OK  airlines_1arima.pkl  ->  1 linha(s)
  OK  airlines_1aspct10.pkl  ->  1 linha(s)
  OK  airlines_1mlppct10.pkl  ->  10 linha(s)
  OK  airlines_1svrpct10.pkl  ->  1 linha(s)
  OK  austres_1amv1pct10.pkl  ->  10 linha(s)
  OK  austres_1arima.pkl  ->  1 linha(s)
  OK  austres_1aspct10.pkl  ->  1 linha(s)
  OK  austres_1mlppct10.pkl  ->  10 linha(s)
  OK  austres_1svrpct10.pkl  ->  1 linha(s)
  OK  coloradoRiver_1amv1pct10.pkl  ->  10 linha(s)
  OK  coloradoRiver_1arima.pkl  ->  1 linha(s)
  OK  coloradoRiver_1aspct10.pkl  ->  1 linha(s)
  OK  coloradoRiver_1mlppct10.pkl  ->  10 linha(s)
  OK  coloradoRiver_1svrpct10.pkl  ->  1 linha(s)
  OK  samurec_1amv1pct10.pkl  ->  10 linha(s)
  OK  samurec_1arima.pkl  ->  1 linha(s)
  OK  samurec_1aspct10.pkl  ->  1 linha(s)
  OK  samurec_1mlppct10.pkl  ->  10 linha(s)
  OK  samurec_1svr

,Serie,Baseline_RMSE,rfecv_RMSE,rfecv_PctGain,rfecv_NFeatures
0,airlines,20.035202,21.074936,-5.189537,12.0
1,austres,18.876899,23.832429,-26.251827,7.0
2,coloradoRiver,0.445994,0.388774,12.829720,45.0
3,samurec,7.264780,7.270354,-0.076733,88.0
4,sunspot,21.135328,23.919068,-13.171028,18.0
5,windspeedfortaleza,0.441626,0.651702,-47.568612,8.0


In [9]:
import json

metadata = {
    'experiment_id': experiment_id,
    'notebook': 'residual_hydridsystem/arima_svr_pct10_rfecv.ipynb',
    'tipo': 'fs pct10',
    'familia': 'ARIMA-SVR hibrido (Additive) / rfecv',
    'janela': 'pct10 -- resolve_lag_size_pct(N - int(config.TEST_SIZE*N), 0.10)',
    'lag_pct_por_serie': {s: lag_pct_por_serie[s] for s in series_list},
    'series': series_list,
    'model_exec': model_exec,
    'seed': None,
    'model_parameters': model_parameters,
    'diff_kpss': experiment_params['diff_kpss'],
    'baseline_pareado': baseline_model_name,
    'strategy': 'rfecv',
}
experiment_dir_results.mkdir(parents=True, exist_ok=True)
(experiment_dir_results / 'metadata.json').write_text(
    json.dumps(metadata, indent=2, default=str), encoding='utf-8'
)
print('metadata.json ->', experiment_dir_results / 'metadata.json')

metadata.json -> C:\Projetos\mestrado_codigos\experiments\results\chamados_pct10_fs_arimasvr_rfecv\metadata.json
